# NER Tagging & Graph RAG for Essential Oils

In [ ]:
import json
from collections import defaultdict

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import spacy

## Load Data

In [ ]:
df = pd.read_csv("filtered_oils_with_shop_urls.csv")
print(f"Loaded {len(df)} essential oils")
df.head()

## Load NER Model

In [ ]:
nlp = spacy.load("de_core_news_sm")

## Extract Named Entities

In [ ]:
def extract_entities(text):
    if pd.isna(text):
        return []
    doc = nlp(str(text))
    return [(ent.text, ent.label_) for ent in doc.ents]

In [ ]:
df["entities"] = df["produktbeschreibung"].apply(extract_entities)
df["benefits_entities"] = df["hauptnutzen"].apply(extract_entities)
df["applications_entities"] = df["anwendungsmoeglichkeiten"].apply(extract_entities)
df["aroma_entities"] = df["aromabeschreibung"].apply(extract_entities)
df["chemical_entities"] = df["hauptchemische_bestandteile"].apply(extract_entities)

print("Entity extraction complete!")

## Build Knowledge Graph

In [ ]:
class KnowledgeGraph:
    def __init__(self):
        self.nodes = set()
        self.edges = defaultdict(list)
        self.entity_types = {}

    def add_node(self, node, entity_type):
        self.nodes.add(node)
        if node not in self.entity_types:
            self.entity_types[node] = entity_type

    def add_edge(self, source, target, relation):
        self.add_node(source, "")
        self.add_node(target, "")
        self.edges[(source, target)].append(relation)

    def add_oil_entity_relation(self, oil_name, entity, entity_type, relation):
        self.add_node(oil_name, "OIL")
        self.add_node(entity, entity_type)
        self.add_edge(oil_name, entity, relation)

    def to_dict(self):
        return {
            "nodes": [{"id": n, "type": self.entity_types.get(n, "UNKNOWN")} for n in self.nodes],
            "edges": [
                {"source": s, "target": t, "relations": r} for (s, t), r in self.edges.items()
            ],
        }

    def to_networkx(self) -> nx.DiGraph:
        G = nx.DiGraph()
        for node in self.nodes:
            G.add_node(node, entity_type=self.entity_types.get(node, "UNKNOWN"))
        for (src, tgt), rels in self.edges.items():
            G.add_edge(src, tgt, relations=rels)
        return G

    def find_oils_by_entity(self, entity: str) -> list[tuple[str, list[str]]]:
        results = []
        for (oil, ent), rels in self.edges.items():
            if entity.lower() in ent.lower():
                results.append((oil, rels))
        return results

    def find_related_entities(self, oil_name: str) -> dict[str, list[str]]:
        related = defaultdict(list)
        for (src, tgt), rels in self.edges.items():
            if src == oil_name:
                related[tgt].extend(rels)
        return dict(related)

    def recommend_similar_oils(self, oil_name: str, max_results: int = 5) -> list[tuple[str, int]]:
        if oil_name not in self.nodes:
            return []

        target_entities = set()
        for (src, tgt), _ in self.edges.items():
            if src == oil_name:
                target_entities.add(tgt)

        scores = defaultdict(int)
        for (src, tgt), _ in self.edges.items():
            if src != oil_name and tgt in target_entities:
                scores[src] += 1

        ranked = sorted(scores.items(), key=lambda x: -x[1])
        return ranked[:max_results]

    def get_entity_centrality(self) -> dict[str, float]:
        G = self.to_networkx()
        return nx.degree_centrality(G)

In [ ]:
kg = KnowledgeGraph()
for _, row in df.iterrows():
    oil_name = row["name"]
    for entity, label in row["entities"]:
        kg.add_oil_entity_relation(oil_name, entity, label, "described_in")
    for entity, label in row["benefits_entities"]:
        kg.add_oil_entity_relation(oil_name, entity, label, "benefits")
    for entity, label in row["applications_entities"]:
        kg.add_oil_entity_relation(oil_name, entity, label, "used_for")
    for entity, label in row["aroma_entities"]:
        kg.add_oil_entity_relation(oil_name, entity, label, "smells_like")
    for entity, label in row["chemical_entities"]:
        kg.add_oil_entity_relation(oil_name, entity, label, "contains")

print(f"Graph: {len(kg.nodes)} nodes, {len(kg.edges)} edges")

In [ ]:
def visualize_graph(kg: KnowledgeGraph, max_nodes: int = 50, figsize: tuple[int, int] = (14, 10)):
    G = kg.to_networkx()

    oils = [n for n, d in G.nodes(data=True) if d.get("entity_type") == "OIL"]
    other_nodes = [n for n in G.nodes if n not in oils]

    sample_oils = oils[: max_nodes // 2]
    sample_entities = [n for n in other_nodes if G.degree(n) >= 2][: max_nodes // 2]
    nodes_to_show = set(sample_oils + sample_entities)

    H = G.subgraph(nodes_to_show).copy()

    plt.figure(figsize=figsize)

    pos = nx.spring_layout(H, k=2, iterations=50, seed=42)

    oil_nodes = [n for n in H.nodes if H.nodes[n].get("entity_type") == "OIL"]
    entity_nodes = [n for n in H.nodes if n not in oil_nodes]

    nx.draw_networkx_nodes(
        H, pos, nodelist=oil_nodes, node_color="#4CAF50", node_size=800, alpha=0.9, label="Oils"
    )
    nx.draw_networkx_nodes(
        H,
        pos,
        nodelist=entity_nodes,
        node_color="#2196F3",
        node_size=400,
        alpha=0.7,
        label="Entities",
    )
    nx.draw_networkx_edges(H, pos, alpha=0.3, edge_color="gray", arrows=True, arrowsize=10)
    nx.draw_networkx_labels(H, pos, font_size=6)

    plt.legend(scatterpoints=1)
    plt.title("Essential Oils Knowledge Graph")
    plt.axis("off")
    plt.tight_layout()
    # plt.savefig("processing/kg_visualization.png", dpi=150, bbox_inches="tight")
    plt.show()
    # print("Saved kg_visualization.png")


def visualize_oil_network(kg: KnowledgeGraph, oil_name: str, max_entities: int = 15):
    G = kg.to_networkx()

    if oil_name not in G:
        print(f"Oil '{oil_name}' not found in graph")
        return

    neighbors = list(G.neighbors(oil_name))[:max_entities]
    nodes = [oil_name] + neighbors
    H = G.subgraph(nodes).copy()

    plt.figure(figsize=(12, 10))
    pos = nx.spring_layout(H, k=2, seed=42)

    nx.draw_networkx_nodes(
        H, pos, nodelist=[oil_name], node_color="#FF5722", node_size=1800, alpha=0.9
    )
    nx.draw_networkx_nodes(
        H, pos, nodelist=neighbors, node_color="#9C27B0", node_size=800, alpha=0.7
    )
    nx.draw_networkx_edges(H, pos, alpha=0.5, edge_color="gray", arrows=True, arrowsize=15)
    nx.draw_networkx_labels(H, pos, font_size=8)

    edge_labels = {}
    for neighbor in neighbors:
        rels = kg.edges.get((oil_name, neighbor), [])
        if rels:
            edge_labels[(oil_name, neighbor)] = ", ".join(set(rels))

    nx.draw_networkx_edge_labels(H, pos, edge_labels, font_size=7, font_color="darkred")

    plt.title(f"Network for {oil_name}\n(edge labels show relation type)", fontsize=12)
    plt.axis("off")
    plt.tight_layout()
    # plt.savefig(
    #     f"processing/{oil_name.replace(' ', '_')}_network.png", dpi=150, bbox_inches="tight"
    # )
    plt.show()
    # print(f"Saved {oil_name}_network.png")

In [ ]:
print("\n1. Full Graph Visualization (sample):")
visualize_graph(kg, max_nodes=800)

sample_oil = df["name"].iloc[0]
print(f"\n2. Single Oil Network ({sample_oil}):")
visualize_oil_network(kg, sample_oil)

print("\n" + "=" * 50)
print("RECOMMENDATION EXAMPLES")
print("=" * 50)

print("\n--- Find oils by entity ---")
results = kg.find_oils_by_entity("Haut")
for oil, rels in results[:3]:
    print(f"  {oil}: {rels}")

print("\n--- Related entities for an oil ---")
sample_oil = df["name"].iloc[0]
related = kg.find_related_entities(sample_oil)
print(f"  {sample_oil} related to:")
for entity, rels in list(related.items())[:5]:
    print(f"    - {entity}: {rels}")

print("\n--- Recommend similar oils ---")
for oil in df["name"].iloc[:3]:
    recs = kg.recommend_similar_oils(oil, max_results=3)
    if recs:
        print(f"  {oil} -> similar: {recs}")

print("\n--- Top entities by centrality ---")
centrality = kg.get_entity_centrality()
top_entities = sorted(
    [(n, c) for n, c in centrality.items() if kg.entity_types.get(n) != "OIL"], key=lambda x: -x[1]
)[:10]
print("  Top entity connections:")
for entity, score in top_entities:
    print(f"    - {entity}: {score:.3f}")

In [ ]:
visualize_oil_network(kg, "Peppermint (Pfefferminze)")

In [ ]:
visualize_oil_network(kg, "Lavender")

## Save Graph

In [ ]:
graph_data = kg.to_dict()
with open("knowledge_graph.json", "w", encoding="utf-8") as f:
    json.dump(graph_data, f, ensure_ascii=False, indent=2)
print("Saved knowledge_graph.json")